# Markdown Parser Research for Thai RAG

This notebook benchmarks Python markdown parsers on the FahMai knowledge base and chooses a primary parser for downstream RAG chunking.

## Evaluation criteria

- Preserve heading hierarchy for section-aware chunks
- Preserve table content for specs and policy rules
- Produce clean flat text for BM25 / embeddings
- Stay reliable over the full corpus
- Keep performance reasonable for notebook iteration

## Sources

- [markdown-it-py usage docs](https://markdown-it-py.readthedocs.io/en/latest/using.html)
- [mdit-py-plugins docs](https://mdit-py-plugins.readthedocs.io/)
- [Mistune 3.2.0 API docs](https://mistune.lepture.com/en/latest/api.html)
- [Marko usage guide](https://marko-py.readthedocs.io/en/latest/usage.html)
- [Python-Markdown extensions docs](https://python-markdown.github.io/extensions/)


In [1]:
from __future__ import annotations

import html
import re
import time
from collections import Counter
from html.parser import HTMLParser
from pathlib import Path
from pprint import pprint
from typing import Any

import mistune
import pandas as pd
from IPython.display import display
from markdown import markdown as python_markdown
from markdown_it import MarkdownIt
from marko import Markdown as MarkoMarkdown
from marko.ast_renderer import ASTRenderer

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)


In [2]:
def resolve_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "knowledge_base").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the competition project directory from the current working directory."
    )


PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data"
KB_DIR = DATA_DIR / "knowledge_base"
QUESTIONS_PATH = DATA_DIR / "questions.csv"

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"KB_DIR: {KB_DIR}")
print(f"QUESTIONS_PATH: {QUESTIONS_PATH}")


PROJECT_DIR: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1
KB_DIR: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/data/knowledge_base
QUESTIONS_PATH: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/data/questions.csv


## 1. Corpus inspection

The benchmark uses the real competition markdown files, not synthetic markdown snippets.


In [3]:
def load_markdown_corpus(root: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for path in sorted(root.rglob("*.md")):
        text = path.read_text(encoding="utf-8")
        records.append(
            {
                "source_path": str(path.relative_to(root.parent)),
                "path": path,
                "doc_type": path.parent.name,
                "filename": path.name,
                "text": text,
                "char_count": len(text),
                "line_count": len(text.splitlines()),
            }
        )
    return records


corpus = load_markdown_corpus(KB_DIR)
print(f"Loaded {len(corpus)} markdown files")

structure_stats = Counter()
for record in corpus:
    text = record["text"]
    structure_stats["files"] += 1
    structure_stats["tables"] += int(bool(re.search(r"^\|.*\|$", text, re.MULTILINE)))
    structure_stats["blockquotes"] += int(bool(re.search(r"^>\s", text, re.MULTILINE)))
    structure_stats["ordered_lists"] += int(
        bool(re.search(r"^\d+\.\s", text, re.MULTILINE))
    )
    structure_stats["unordered_lists"] += int(
        bool(re.search(r"^[-*+]\s", text, re.MULTILINE))
    )
    structure_stats["headings"] += int(
        bool(re.search(r"^#{1,6}\s", text, re.MULTILINE))
    )
    structure_stats["horizontal_rules"] += int(
        bool(re.search(r"^---+$", text, re.MULTILINE))
    )

pd.DataFrame([structure_stats]).T.rename(columns={0: "count"})


Loaded 118 markdown files


,count
files,118
tables,117
blockquotes,24
ordered_lists,5
unordered_lists,118
headings,118
horizontal_rules,86


In [4]:
corpus_df = pd.DataFrame(
    [
        {
            "source_path": record["source_path"],
            "doc_type": record["doc_type"],
            "char_count": record["char_count"],
            "line_count": record["line_count"],
        }
        for record in corpus
    ]
)

corpus_df.groupby("doc_type").agg(
    files=("source_path", "count"),
    avg_chars=("char_count", "mean"),
    avg_lines=("line_count", "mean"),
).round(1)


,files,avg_chars,avg_lines
doc_type,,,
policies,5,6576.0,173.2
products,110,2953.9,78.5
store_info,3,8305.3,177.3


## 2. Parser candidates and notebook assumptions

The comparison focuses on four libraries:

- `markdown-it-py`: token-stream oriented and strong for section-aware chunking
- `mistune`: fast baseline with AST output and plugins
- `marko`: AST-oriented and CommonMark-aligned
- `Python-Markdown`: mature extension-based parser, mostly HTML-oriented


In [5]:
class HTMLTextExtractor(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.parts: list[str] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag in {
            "p",
            "div",
            "section",
            "article",
            "table",
            "tr",
            "ul",
            "ol",
            "li",
            "blockquote",
            "br",
            "h1",
            "h2",
            "h3",
            "h4",
            "h5",
            "h6",
        }:
            self.parts.append("\n")

    def handle_data(self, data: str) -> None:
        if data:
            self.parts.append(data)

    def handle_endtag(self, tag: str) -> None:
        if tag in {
            "p",
            "div",
            "section",
            "article",
            "table",
            "tr",
            "ul",
            "ol",
            "li",
            "blockquote",
            "h1",
            "h2",
            "h3",
            "h4",
            "h5",
            "h6",
        }:
            self.parts.append("\n")

    def get_text(self) -> str:
        return normalize_whitespace("".join(self.parts))


def normalize_whitespace(text: str) -> str:
    text = html.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def html_to_text(text: str) -> str:
    parser = HTMLTextExtractor()
    parser.feed(text)
    return parser.get_text()


def flatten_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(part for part in (flatten_text(item) for item in value) if part)
    if isinstance(value, dict):
        for key in ("raw", "content", "text"):
            if isinstance(value.get(key), str):
                return value[key]
        return " ".join(
            part
            for part in (flatten_text(item) for item in value.get("children", []))
            if part
        )
    return str(value)


def extract_markdown_tables(text: str) -> list[str]:
    blocks: list[str] = []
    current: list[str] = []
    for line in text.splitlines():
        if line.strip().startswith("|") and line.strip().endswith("|"):
            current.append(line.rstrip())
        else:
            if len(current) >= 2:
                blocks.append("\n".join(current))
            current = []
    if len(current) >= 2:
        blocks.append("\n".join(current))
    return blocks


def make_chunk(
    source_path: str,
    doc_type: str,
    heading_path: list[str],
    chunk_text: str,
    chunk_order: int,
    contains_table: bool,
    parser_name: str,
) -> dict[str, Any]:
    return {
        "source_path": source_path,
        "doc_type": doc_type,
        "heading_path": " > ".join(heading_path),
        "chunk_text": normalize_whitespace(chunk_text),
        "chunk_order": chunk_order,
        "contains_table": contains_table,
        "parser_name": parser_name,
    }


## 3. Shared normalization helpers

Each parser adapter returns the same normalized record:

- `source_path`
- `parser_name`
- `plain_text`
- `headings`
- `tables`
- `blocks`
- `metadata`
- `parse_time_ms`
- `error`


In [6]:
def finalize_parse_result(
    source_path: str,
    parser_name: str,
    doc_type: str,
    blocks: list[dict[str, Any]],
    headings: list[str],
    tables: list[str],
    parse_time_ms: float,
    error: str | None = None,
    metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    clean_blocks = []
    for block in blocks:
        text = normalize_whitespace(block.get("text", ""))
        if not text:
            continue
        clean_blocks.append(
            {
                "block_type": block.get("block_type", "unknown"),
                "heading_path": block.get("heading_path", []),
                "text": text,
                "contains_table": bool(block.get("contains_table", False)),
            }
        )

    plain_text = normalize_whitespace(
        "\n\n".join(block["text"] for block in clean_blocks)
    )
    return {
        "source_path": source_path,
        "parser_name": parser_name,
        "plain_text": plain_text,
        "headings": headings,
        "tables": tables,
        "blocks": clean_blocks,
        "metadata": metadata or {},
        "parse_time_ms": round(parse_time_ms, 3),
        "error": error,
        "doc_type": doc_type,
    }


def empty_parse_result(
    source_path: str, parser_name: str, doc_type: str, error: str
) -> dict[str, Any]:
    return {
        "source_path": source_path,
        "parser_name": parser_name,
        "plain_text": "",
        "headings": [],
        "tables": [],
        "blocks": [],
        "metadata": {},
        "parse_time_ms": None,
        "error": error,
        "doc_type": doc_type,
    }


def to_rag_chunks(parsed_doc: dict[str, Any]) -> list[dict[str, Any]]:
    chunks: list[dict[str, Any]] = []
    for index, block in enumerate(parsed_doc["blocks"]):
        chunks.append(
            make_chunk(
                source_path=parsed_doc["source_path"],
                doc_type=parsed_doc["doc_type"],
                heading_path=block["heading_path"],
                chunk_text=block["text"],
                chunk_order=index,
                contains_table=block["contains_table"],
                parser_name=parsed_doc["parser_name"],
            )
        )
    return chunks


## 4. Parser adapters


In [7]:
def parse_with_markdown_it(text: str, source_path: str) -> dict[str, Any]:
    start = time.perf_counter()
    md = MarkdownIt("commonmark").enable("table").enable("strikethrough")
    lines = text.splitlines()
    tokens = md.parse(text)
    stack: list[str | None] = []
    headings: list[str] = []
    tables: list[str] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.type == "heading_open":
            level = int(token.tag[1])
            inline_token = tokens[i + 1] if i + 1 < len(tokens) else None
            heading_text = normalize_whitespace(getattr(inline_token, "content", ""))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            headings.append(heading_text)
            i += 1
            continue
        if token.type in {
            "paragraph_open",
            "blockquote_open",
            "bullet_list_open",
            "ordered_list_open",
            "table_open",
        }:
            start_line, end_line = token.map or (None, None)
            segment = (
                "\n".join(lines[start_line:end_line])
                if start_line is not None and end_line is not None
                else ""
            )
            if segment:
                contains_table = token.type == "table_open"
                if contains_table:
                    tables.append(segment)
                blocks.append(
                    {
                        "block_type": token.type.removesuffix("_open"),
                        "heading_path": current_path(),
                        "text": segment,
                        "contains_table": contains_table,
                    }
                )
        elif token.type == "fence" and token.content:
            blocks.append(
                {
                    "block_type": "fence",
                    "heading_path": current_path(),
                    "text": token.content,
                    "contains_table": False,
                }
            )
        i += 1

    if not tables:
        tables = extract_markdown_tables(text)

    return finalize_parse_result(
        source_path=source_path,
        parser_name="markdown-it-py",
        doc_type=Path(source_path).parts[1],
        blocks=blocks,
        headings=headings,
        tables=tables,
        parse_time_ms=(time.perf_counter() - start) * 1000,
        metadata={"token_count": len(tokens)},
    )


In [8]:
def render_mistune_table(node: dict[str, Any]) -> str:
    rows: list[str] = []
    for section in node.get("children", []):
        for row in section.get("children", []):
            cells = [
                normalize_whitespace(flatten_text(cell))
                for cell in row.get("children", [])
            ]
            rows.append(" | ".join(cells))
    return "\n".join(row for row in rows if row)


def parse_with_mistune(text: str, source_path: str) -> dict[str, Any]:
    start = time.perf_counter()
    md = mistune.create_markdown(
        renderer="ast", plugins=["table", "url", "strikethrough"]
    )
    ast = md(text)
    stack: list[str | None] = []
    headings: list[str] = []
    tables: list[str] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    for node in ast:
        node_type = node.get("type")
        if node_type == "heading":
            level = node.get("attrs", {}).get("level", 1)
            heading_text = normalize_whitespace(flatten_text(node.get("children", [])))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            headings.append(heading_text)
            continue
        if node_type == "blank_line":
            continue
        if node_type == "table":
            table_text = render_mistune_table(node)
            tables.append(table_text)
            blocks.append(
                {
                    "block_type": "table",
                    "heading_path": current_path(),
                    "text": table_text,
                    "contains_table": True,
                }
            )
            continue
        block_text = normalize_whitespace(flatten_text(node.get("children", [])))
        if block_text:
            blocks.append(
                {
                    "block_type": node_type,
                    "heading_path": current_path(),
                    "text": block_text,
                    "contains_table": False,
                }
            )

    if not tables:
        tables = extract_markdown_tables(text)

    return finalize_parse_result(
        source_path=source_path,
        parser_name="mistune",
        doc_type=Path(source_path).parts[1],
        blocks=blocks,
        headings=headings,
        tables=tables,
        parse_time_ms=(time.perf_counter() - start) * 1000,
        metadata={"root_nodes": len(ast)},
    )


In [9]:
def flatten_marko_ast(node: Any) -> str:
    if isinstance(node, str):
        return node
    if isinstance(node, list):
        return " ".join(
            part for part in (flatten_marko_ast(item) for item in node) if part
        )
    if isinstance(node, dict):
        if isinstance(node.get("children"), list):
            return " ".join(
                part
                for part in (flatten_marko_ast(item) for item in node["children"])
                if part
            )
        if isinstance(node.get("children"), str):
            return node["children"]
        if isinstance(node.get("body"), str):
            return node["body"]
        if isinstance(node.get("title"), str):
            return node["title"]
    return ""


def render_marko_table(node: dict[str, Any]) -> str:
    rows: list[str] = []
    for child in node.get("children", []):
        child_element = child.get("element")
        if child_element == "table_row":
            nested_rows = [child]
        elif child_element in {"table_body", "table_head"}:
            nested_rows = child.get("children", [])
        else:
            continue
        for row in nested_rows:
            if row.get("element") != "table_row":
                continue
            cells = [
                normalize_whitespace(flatten_marko_ast(cell))
                for cell in row.get("children", [])
            ]
            rows.append(" | ".join(cells))
    return "\n".join(row for row in rows if row)


def parse_with_marko(text: str, source_path: str) -> dict[str, Any]:
    start = time.perf_counter()
    md = MarkoMarkdown(renderer=ASTRenderer, extensions=["gfm"])
    ast = md(text)
    stack: list[str | None] = []
    headings: list[str] = []
    tables: list[str] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    for node in ast.get("children", []):
        element = node.get("element")
        if element == "heading":
            level = node.get("level", 1)
            heading_text = normalize_whitespace(
                flatten_marko_ast(node.get("children", []))
            )
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            headings.append(heading_text)
            continue
        if element == "table":
            table_text = render_marko_table(node)
            tables.append(table_text)
            blocks.append(
                {
                    "block_type": "table",
                    "heading_path": current_path(),
                    "text": table_text,
                    "contains_table": True,
                }
            )
            continue
        block_text = normalize_whitespace(flatten_marko_ast(node.get("children", [])))
        if block_text:
            blocks.append(
                {
                    "block_type": element or "unknown",
                    "heading_path": current_path(),
                    "text": block_text,
                    "contains_table": False,
                }
            )

    if not tables:
        tables = extract_markdown_tables(text)

    return finalize_parse_result(
        source_path=source_path,
        parser_name="marko",
        doc_type=Path(source_path).parts[1],
        blocks=blocks,
        headings=headings,
        tables=tables,
        parse_time_ms=(time.perf_counter() - start) * 1000,
        metadata={"root_children": len(ast.get("children", []))},
    )


In [10]:
def parse_with_python_markdown(text: str, source_path: str) -> dict[str, Any]:
    start = time.perf_counter()
    html_output = python_markdown(
        text, extensions=["tables", "fenced_code", "sane_lists", "nl2br"]
    )
    plain_text = html_to_text(html_output)
    headings = [
        match.group(2).strip()
        for match in re.finditer(r"^(#{1,6})\s+(.*)$", text, flags=re.MULTILINE)
    ]
    tables = extract_markdown_tables(text)

    blocks: list[dict[str, Any]] = []
    current_heading_path: list[str] = []
    buffer: list[str] = []

    def flush_buffer() -> None:
        block_text = normalize_whitespace("\n".join(buffer))
        if block_text:
            blocks.append(
                {
                    "block_type": "heuristic_block",
                    "heading_path": current_heading_path.copy(),
                    "text": block_text,
                    "contains_table": any("|" in line for line in buffer),
                }
            )
        buffer.clear()

    stack: list[str | None] = []
    for line in text.splitlines():
        heading_match = re.match(r"^(#{1,6})\s+(.*)$", line)
        if heading_match:
            flush_buffer()
            level = len(heading_match.group(1))
            title = heading_match.group(2).strip()
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = title
            stack[:] = stack[:level]
            current_heading_path = [item for item in stack if item]
            continue
        if not line.strip():
            flush_buffer()
            continue
        buffer.append(line)
    flush_buffer()

    result = finalize_parse_result(
        source_path=source_path,
        parser_name="python-markdown",
        doc_type=Path(source_path).parts[1],
        blocks=blocks,
        headings=headings,
        tables=tables,
        parse_time_ms=(time.perf_counter() - start) * 1000,
        metadata={"html_length": len(html_output)},
    )
    result["plain_text"] = plain_text
    return result


In [11]:
PARSERS = {
    "markdown-it-py": parse_with_markdown_it,
    "mistune": parse_with_mistune,
    "marko": parse_with_marko,
    "python-markdown": parse_with_python_markdown,
}


def safe_parse(parser_name: str, text: str, source_path: str) -> dict[str, Any]:
    doc_type = Path(source_path).parts[1]
    try:
        return PARSERS[parser_name](text, source_path)
    except Exception as exc:
        return empty_parse_result(source_path, parser_name, doc_type, repr(exc))


print("Parsers configured:")
pprint(list(PARSERS))


Parsers configured:
['markdown-it-py', 'mistune', 'marko', 'python-markdown']


## 5. Qualitative output comparison

Compare one product document, one policy document, and one FAQ-like document side by side.


In [12]:
SAMPLE_DOCS = [
    "knowledge_base/products/DN-LT-002_daonuea_airbook_14.md",
    "knowledge_base/policies/shipping_policy.md",
    "knowledge_base/store_info/general_faq.md",
]

sample_records = {
    record["source_path"]: record
    for record in corpus
    if record["source_path"] in SAMPLE_DOCS
}
list(sample_records)


['knowledge_base/policies/shipping_policy.md',
 'knowledge_base/products/DN-LT-002_daonuea_airbook_14.md',
 'knowledge_base/store_info/general_faq.md']

In [13]:
def compare_single_document(source_path: str) -> pd.DataFrame:
    record = sample_records[source_path]
    rows = []
    for parser_name in PARSERS:
        parsed = safe_parse(parser_name, record["text"], record["source_path"])
        rows.append(
            {
                "parser": parser_name,
                "error": parsed["error"],
                "source_path": source_path,
                "parse_time_ms": parsed["parse_time_ms"],
                "heading_count": len(parsed["headings"]),
                "table_count": len(parsed["tables"]),
                "block_count": len(parsed["blocks"]),
                "plain_text_preview": parsed["plain_text"][:260],
                "first_heading_path": " > ".join(parsed["blocks"][0]["heading_path"])
                if parsed["blocks"]
                else "",
                "first_block_preview": parsed["blocks"][0]["text"][:220]
                if parsed["blocks"]
                else "",
            }
        )
    return pd.DataFrame(rows)


for source_path in SAMPLE_DOCS:
    display(compare_single_document(source_path))


,parser,error,source_path,parse_time_ms,heading_count,table_count,block_count,plain_text_preview,first_heading_path,first_block_preview
0,markdown-it-py,None,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,4.722,8,1,36,"รหัสสินค้า: DN-LT-002\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\nหมวดหมู่: แล็ปท็อป\nราคา: ฿29,990\nสถานะ: มีสินค้า\nวันที่อัปเ...",ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14),"รหัสสินค้า: DN-LT-002\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\nหมวดหมู่: แล็ปท็อป\nราคา: ฿29,990\nสถานะ: มีสินค้า\nวันที่อัปเ..."
1,mistune,None,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,12.319,8,1,20,"รหัสสินค้า: DN-LT-002 แบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่ หมวดหมู่: แล็ปท็อป ราคา: ฿29,990 สถานะ: มีสินค้า วันที่อัปเดต: 1...",ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14),"รหัสสินค้า: DN-LT-002 แบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่ หมวดหมู่: แล็ปท็อป ราคา: ฿29,990 สถานะ: มีสินค้า วันที่อัปเดต: 1..."
2,marko,None,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,21.373,8,1,20,"รหัสสินค้า: DN-LT-002 \n แบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่ \n หมวดหมู่: แล็ปท็อป \n ราคา: ฿29,990 \n สถานะ: มีสินค้า \n ...",ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14),"รหัสสินค้า: DN-LT-002 \n แบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่ \n หมวดหมู่: แล็ปท็อป \n ราคา: ฿29,990 \n สถานะ: มีสินค้า \n ..."
3,python-markdown,None,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,25.892,8,1,18,ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14)\n\nรหัสสินค้า: DN-LT-002\n\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\n\nหมวดหมู่: แล็...,ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14),"รหัสสินค้า: DN-LT-002\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\nหมวดหมู่: แล็ปท็อป\nราคา: ฿29,990\nสถานะ: มีสินค้า\nวันที่อัปเ..."


,parser,error,source_path,parse_time_ms,heading_count,table_count,block_count,plain_text_preview,first_heading_path,first_block_preview
0,markdown-it-py,None,knowledge_base/policies/shipping_policy.md,3.947,17,4,58,**วันที่อัปเดต:** 1 มีนาคม 2569\n\nฟ้าใหม่จัดส่งสินค้าทั่วประเทศไทย ผ่านบริษัทขนส่งพาร์ทเนอร์ที่คัดเลือกแล้ว โดยมีการติดตาม (Tracking) ท...,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่,**วันที่อัปเดต:** 1 มีนาคม 2569
1,mistune,None,knowledge_base/policies/shipping_policy.md,3.980,17,4,33,วันที่อัปเดต: 1 มีนาคม 2569\n\nฟ้าใหม่จัดส่งสินค้าทั่วประเทศไทย ผ่านบริษัทขนส่งพาร์ทเนอร์ที่คัดเลือกแล้ว โดยมีการติดตาม (Tracking) ทุกคำ...,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569
2,marko,None,knowledge_base/policies/shipping_policy.md,15.619,17,3,32,วันที่อัปเดต: 1 มีนาคม 2569\n\nฟ้าใหม่จัดส่งสินค้าทั่วประเทศไทย ผ่านบริษัทขนส่งพาร์ทเนอร์ที่คัดเลือกแล้ว โดยมีการติดตาม (Tracking) ทุกคำ...,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569
3,python-markdown,None,knowledge_base/policies/shipping_policy.md,4.859,17,4,39,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่\n\nวันที่อัปเดต: 1 มีนาคม 2569\n\n1. ภาพรวมการจัดส่ง\n\nฟ้าใหม่จัดส่งสินค้าทั่วประเทศไทย ผ่านบริษัทข...,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่,**วันที่อัปเดต:** 1 มีนาคม 2569


,parser,error,source_path,parse_time_ms,heading_count,table_count,block_count,plain_text_preview,first_heading_path,first_block_preview
0,markdown-it-py,None,knowledge_base/store_info/general_faq.md,11.700,9,1,43,วันที่อัปเดต: 1 มีนาคม 2569\n\n**Q: สั่งซื้อสินค้าได้ที่ไหนบ้าง?**\nสามารถสั่งซื้อสินค้าได้ 3 ช่องทางหลัก ได้แก่ เว็บไซต์ www.fahmai.co....,คำถามที่พบบ่อย (General FAQ) — ฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569
1,mistune,None,knowledge_base/store_info/general_faq.md,5.576,9,1,38,วันที่อัปเดต: 1 มีนาคม 2569\n\nQ: สั่งซื้อสินค้าได้ที่ไหนบ้าง? สามารถสั่งซื้อสินค้าได้ 3 ช่องทางหลัก ได้แก่ เว็บไซต์ www.fahmai.co.th แอ...,คำถามที่พบบ่อย (General FAQ) — ฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569
2,marko,None,knowledge_base/store_info/general_faq.md,18.210,9,1,38,วันที่อัปเดต: 1 มีนาคม 2569\n\nQ: สั่งซื้อสินค้าได้ที่ไหนบ้าง? \n สามารถสั่งซื้อสินค้าได้ 3 ช่องทางหลัก ได้แก่ เว็บไซต์ www.fahmai.co.th...,คำถามที่พบบ่อย (General FAQ) — ฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569
3,python-markdown,None,knowledge_base/store_info/general_faq.md,4.893,9,1,72,คำถามที่พบบ่อย (General FAQ) — ฟ้าใหม่\n\nวันที่อัปเดต: 1 มีนาคม 2569\n\nหมวด 1: การสั่งซื้อสินค้า\n\nQ: สั่งซื้อสินค้าได้ที่ไหนบ้าง?\n\...,คำถามที่พบบ่อย (General FAQ) — ฟ้าใหม่,วันที่อัปเดต: 1 มีนาคม 2569


In [14]:
def show_parser_chunks(
    source_path: str, parser_name: str = "markdown-it-py", n: int = 5
) -> pd.DataFrame:
    record = sample_records[source_path]
    parsed = safe_parse(parser_name, record["text"], record["source_path"])
    return pd.DataFrame(to_rag_chunks(parsed)).head(n)


show_parser_chunks(
    "knowledge_base/policies/shipping_policy.md", parser_name="markdown-it-py", n=8
)


,source_path,doc_type,heading_path,chunk_text,chunk_order,contains_table,parser_name
0,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่,**วันที่อัปเดต:** 1 มีนาคม 2569,0,False,markdown-it-py
1,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 1. ภาพรวมการจัดส่ง,ฟ้าใหม่จัดส่งสินค้าทั่วประเทศไทย ผ่านบริษัทขนส่งพาร์ทเนอร์ที่คัดเลือกแล้ว โดยมีการติดตาม (Tracking) ทุกคำสั่งซื้อแบบ Real-time สินค้าทุก...,1,False,markdown-it-py
2,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 1. ภาพรวมการจัดส่ง,**วันทำการ:** จันทร์ – เสาร์ (ไม่รวมวันอาทิตย์และวันหยุดนักขัตฤกษ์)\nคำสั่งซื้อที่ได้รับก่อน 12:00 น. ในวันทำการ จะถูกประมวลผลในวันเดียว...,2,False,markdown-it-py
3,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,| มูลค่าคำสั่งซื้อ | ค่าจัดส่ง | หมายเหตุ |\n|----------------|----------|----------|\n| ฿500 ขึ้นไป | **ฟรี** | จัดส่งมาตรฐานทั่วประเทศ...,3,True,markdown-it-py
4,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,"> **ตัวอย่าง:**\n> - สั่งสินค้า ฿450 → ค่าจัดส่งมาตรฐาน ฿50\n> - สั่งสินค้า ฿500 → จัดส่งฟรี\n> - สั่งสินค้า ฿1,200 + เลือก Express ในกร...",4,False,markdown-it-py
5,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,> **ตัวอย่าง:**,5,False,markdown-it-py
6,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,"> - สั่งสินค้า ฿450 → ค่าจัดส่งมาตรฐาน ฿50\n> - สั่งสินค้า ฿500 → จัดส่งฟรี\n> - สั่งสินค้า ฿1,200 + เลือก Express ในกรุงเทพฯ → ค่าจัดส่...",6,False,markdown-it-py
7,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,> - สั่งสินค้า ฿450 → ค่าจัดส่งมาตรฐาน ฿50,7,False,markdown-it-py


## 6. Quantitative benchmark


In [15]:
def benchmark_parsers(
    files: list[dict[str, Any]], parsers: dict[str, Any]
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for parser_name in parsers:
        parser_docs = []
        total_start = time.perf_counter()
        for record in files:
            parser_docs.append(
                safe_parse(parser_name, record["text"], record["source_path"])
            )
        total_elapsed_ms = (time.perf_counter() - total_start) * 1000

        parse_times = [
            doc["parse_time_ms"]
            for doc in parser_docs
            if doc["parse_time_ms"] is not None
        ]
        rows.append(
            {
                "parser": parser_name,
                "docs": len(parser_docs),
                "errors": sum(1 for doc in parser_docs if doc["error"]),
                "docs_with_headings": sum(1 for doc in parser_docs if doc["headings"]),
                "docs_with_tables": sum(1 for doc in parser_docs if doc["tables"]),
                "avg_parse_ms": round(pd.Series(parse_times).mean(), 3)
                if parse_times
                else None,
                "total_parse_ms": round(total_elapsed_ms, 3),
                "avg_blocks": round(
                    pd.Series([len(doc["blocks"]) for doc in parser_docs]).mean(), 2
                ),
                "avg_chunks": round(
                    pd.Series([len(to_rag_chunks(doc)) for doc in parser_docs]).mean(),
                    2,
                ),
                "avg_heading_count": round(
                    pd.Series([len(doc["headings"]) for doc in parser_docs]).mean(), 2
                ),
                "avg_table_count": round(
                    pd.Series([len(doc["tables"]) for doc in parser_docs]).mean(), 2
                ),
                "avg_plain_text_chars": round(
                    pd.Series([len(doc["plain_text"]) for doc in parser_docs]).mean(), 1
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["errors", "avg_parse_ms", "avg_blocks"], na_position="last"
    )


benchmark_df = benchmark_parsers(corpus, PARSERS)
benchmark_df


,parser,docs,errors,docs_with_headings,docs_with_tables,avg_parse_ms,total_parse_ms,avg_blocks,avg_chunks,avg_heading_count,avg_table_count,avg_plain_text_chars
0,markdown-it-py,118,0,118,117,1.483,206.195,28.40,28.40,7.5,1.18,3632.5
1,mistune,118,0,118,117,1.520,199.462,15.74,15.74,7.5,1.18,2851.7
3,python-markdown,118,0,118,117,2.496,317.198,19.03,19.03,7.5,1.18,3047.1
2,marko,118,0,118,117,5.048,617.865,15.73,15.73,7.5,1.17,2868.0


In [16]:
def microbenchmark_document(source_path: str, iterations: int = 40) -> pd.DataFrame:
    record = sample_records[source_path]
    rows = []
    for parser_name in PARSERS:
        start = time.perf_counter()
        for _ in range(iterations):
            safe_parse(parser_name, record["text"], record["source_path"])
        elapsed_ms = (time.perf_counter() - start) * 1000
        rows.append(
            {
                "parser": parser_name,
                "source_path": source_path,
                "iterations": iterations,
                "total_ms": round(elapsed_ms, 3),
                "avg_ms": round(elapsed_ms / iterations, 3),
            }
        )
    return pd.DataFrame(rows).sort_values("avg_ms")


microbenchmark_document(
    "knowledge_base/products/DN-LT-002_daonuea_airbook_14.md", iterations=60
)


,parser,source_path,iterations,total_ms,avg_ms
0,markdown-it-py,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,60,107.371,1.790
1,mistune,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,60,129.880,2.165
3,python-markdown,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,60,204.118,3.402
2,marko,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,60,364.633,6.077


## 7. Retrieval-oriented sanity checks

These checks are intentionally simple. The goal is to confirm that a parser can produce chunks useful for questions about price, Bluetooth version, warranty, shipping, and policy edge cases.


In [17]:
questions_df = pd.read_csv(QUESTIONS_PATH)
questions_df.head(5)


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
3,4,อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้าใหม่ได้ไหม?,ได้ ลดราคาสูงสุด 30%,ได้ เฉพาะสมาร์ทโฟนสายฟ้าเท่านั้น,ได้ ต้องเป็นสมาชิก Gold ขึ้นไป,ได้ เฉพาะสินค้าที่ซื้อจากฟ้าใหม่,"ได้ แต่ต้องซื้อเครื่องใหม่ราคาเกิน 10,000 บาท",ไม่มีบริการ Trade-in ในขณะนี้,ได้ เฉพาะช่วง Mega Sale,ได้ ต้องนำไปที่สาขาเท่านั้น,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
4,5,สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ,ไม่จำกัดจำนวน,5 รายการ,20 รายการ,15 รายการ,3 รายการ,10 รายการ,50 รายการ,8 รายการ,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


In [18]:
def find_matching_chunks(
    parser_name: str, query_terms: list[str], limit: int = 8
) -> pd.DataFrame:
    all_chunks: list[dict[str, Any]] = []
    for record in corpus:
        parsed = safe_parse(parser_name, record["text"], record["source_path"])
        all_chunks.extend(to_rag_chunks(parsed))

    def score(text: str) -> int:
        lowered = text.lower()
        return sum(term.lower() in lowered for term in query_terms)

    scored = []
    for chunk in all_chunks:
        chunk_score = score(chunk["chunk_text"])
        if chunk_score:
            scored.append({**chunk, "match_score": chunk_score})

    scored_df = pd.DataFrame(scored)
    if scored_df.empty:
        return scored_df
    return scored_df.sort_values(
        ["match_score", "contains_table", "source_path"], ascending=[False, False, True]
    ).head(limit)


find_matching_chunks("markdown-it-py", ["Bluetooth 5.3", "AirBook 14"])


,source_path,doc_type,heading_path,chunk_text,chunk_order,contains_table,parser_name,match_score
0,knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md,products,ดาวเหนือ Tower X10 Max (DaoNuea Tower X10 Max) > สเปคสินค้า,| รายการ | ข้อมูล |\n|--------|--------|\n| CPU | DaoNuea Octa-Core 4.2GHz (Turbo 5.5GHz) |\n| GPU | StormForce GX-4090 (16GB GDDR6X) |\...,5,True,markdown-it-py,1
1,knowledge_base/products/DN-LT-001_daonuea_airbook_15.md,products,ดาวเหนือ แอร์บุ๊ก 15 (DaoNuea AirBook 15) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 15.6 นิ้ว IPS, 2560×1600 (2K), 60Hz, sRGB 100%, Anti-glare |\n| ชิปเซ็ต | Da...",4,True,markdown-it-py,1
6,knowledge_base/products/DN-LT-002_daonuea_airbook_14.md,products,ดาวเหนือ แอร์บุ๊ก 14 (DaoNuea AirBook 14) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 14 นิ้ว IPS, 2560×1600 (2K), 60Hz, sRGB 100%, ความสว่าง 400 nit |\n| ชิปเซ็ต...",6,True,markdown-it-py,1
19,knowledge_base/products/DN-LT-003_daonuea_airbook_14_8gb.md,products,ดาวเหนือ แอร์บุ๊ก 14 รุ่น 8GB (DaoNuea AirBook 14 8GB) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 14 นิ้ว IPS, 2560×1600 (2K), 60Hz, sRGB 100%, ความสว่าง 400 nit |\n| ชิปเซ็ต...",6,True,markdown-it-py,1
26,knowledge_base/products/DN-LT-004_daonuea_probook_16.md,products,ดาวเหนือ โปรบุ๊ก 16 (DaoNuea ProBook 16) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 16 นิ้ว mini-LED, 3456×2160 (3.5K), 120Hz, DCI-P3 98%, ความสว่างพีค 1,000 ni...",7,True,markdown-it-py,1
27,knowledge_base/products/DN-LT-005_daonuea_probook_14.md,products,ดาวเหนือ โปรบุ๊ก 14 (DaoNuea ProBook 14) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 14 นิ้ว mini-LED, 2880×1800 (2.8K), 120Hz, DCI-P3 95%, ความสว่างพีค 800 nit,...",5,True,markdown-it-py,1
28,knowledge_base/products/DN-LT-006_daonuea_probook_14_max.md,products,ดาวเหนือ โปรบุ๊ก 14 แม็กซ์ (DaoNuea ProBook 14 Max) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 14 นิ้ว mini-LED, 2880×1800 (2.8K), 120Hz, DCI-P3 95%, ความสว่างพีค 800 nit,...",7,True,markdown-it-py,1
30,knowledge_base/products/DN-LT-007_daonuea_stormbook_g7.md,products,แล็ปท็อปเกมมิ่ง ดาวเหนือ สตอร์มบุ๊ก G7 (DaoNuea StormBook G7) > สเปคสินค้า,"| รายการ | รายละเอียด |\n|--------|-----------|\n| หน้าจอ | 16 นิ้ว IPS, Full HD (1920×1200), 144Hz, sRGB 100% |\n| ชิปเซ็ต (CPU) | ดาวเ...",5,True,markdown-it-py,1


In [19]:
find_matching_chunks("markdown-it-py", ["Express", "Platinum", "ฟรี"])


,source_path,doc_type,heading_path,chunk_text,chunk_order,contains_table,parser_name,match_score
4,knowledge_base/policies/membership_points_policy.md,policies,นโยบายสมาชิกและ FahMai Points — ร้านฟ้าใหม่ > 2. ระดับสมาชิก (Membership Tiers) > 2.3 สมาชิก Platinum,"| รายการ | รายละเอียด |\n|--------|-----------|\n| **เงื่อนไข** | ยอดสะสม ฿50,000 ขึ้นไป |\n| **อัตราสะสม Points** | **x2** (เก็บได้มากก...",9,True,markdown-it-py,3
19,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง > 2.1 สมาชิก Gold และ Platinum,"- **Gold (ยอดสะสม ฿10,000–฿49,999):** จัดส่งฟรีทุกคำสั่งซื้อ ไม่มีขั้นต่ำ\n- **Platinum (ยอดสะสม ฿50,000 ขึ้นไป):** จัดส่งฟรี + Express ...",10,False,markdown-it-py,3
21,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง > 2.1 สมาชิก Gold และ Platinum,"- **Platinum (ยอดสะสม ฿50,000 ขึ้นไป):** จัดส่งฟรี + Express ฟรี (ในเขตกรุงเทพฯ+ปริมณฑล)",12,False,markdown-it-py,3
23,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น),เพิ่มค่าบริการ +฿100 (ยกเว้นสมาชิก Platinum ซึ่ง Express ฟรี),23,False,markdown-it-py,3
14,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,| มูลค่าคำสั่งซื้อ | ค่าจัดส่ง | หมายเหตุ |\n|----------------|----------|----------|\n| ฿500 ขึ้นไป | **ฟรี** | จัดส่งมาตรฐานทั่วประเทศ...,3,True,markdown-it-py,2
15,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,"> **ตัวอย่าง:**\n> - สั่งสินค้า ฿450 → ค่าจัดส่งมาตรฐาน ฿50\n> - สั่งสินค้า ฿500 → จัดส่งฟรี\n> - สั่งสินค้า ฿1,200 + เลือก Express ในกร...",4,False,markdown-it-py,2
16,knowledge_base/policies/shipping_policy.md,policies,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,"> - สั่งสินค้า ฿450 → ค่าจัดส่งมาตรฐาน ฿50\n> - สั่งสินค้า ฿500 → จัดส่งฟรี\n> - สั่งสินค้า ฿1,200 + เลือก Express ในกรุงเทพฯ → ค่าจัดส่...",6,False,markdown-it-py,2
2,knowledge_base/policies/membership_points_policy.md,policies,นโยบายสมาชิกและ FahMai Points — ร้านฟ้าใหม่ > 2. ระดับสมาชิก (Membership Tiers) > 2.1 สมาชิก Silver,"| รายการ | รายละเอียด |\n|--------|-----------|\n| **เงื่อนไข** | ยอดสะสม ฿0 – ฿9,999 |\n| **อัตราสะสม Points** | 1 Point ต่อการใช้จ่าย ...",4,True,markdown-it-py,1


best_parser = scored_benchmark_df.iloc[0]["parser"]
runner_up = scored_benchmark_df.iloc[1]["parser"]

recommendation_lines = [
    f"Recommended primary parser: {best_parser}",
    f"Runner-up: {runner_up}",
    "",
    "Decision notes:",
    "- Prefer markdown-it-py if it preserves headings and tables cleanly with acceptable runtime.",
    "- Keep mistune as the fallback when iteration speed matters more than the richest block structure.",
    "- Use marko if its AST proves easier to maintain than token-walking in later RAG notebooks.",
    "- Keep Python-Markdown as a reference parser, not the default RAG parser, unless it clearly wins on this corpus.",
]

print("\n".join(recommendation_lines))


In [20]:
def score_parser_row(row: pd.Series) -> float:
    avg_parse_ms = row["avg_parse_ms"] if row["avg_parse_ms"] is not None else 1_000_000
    score = 0.0
    score += row["docs_with_headings"] / len(corpus) * 3.0
    score += row["docs_with_tables"] / len(corpus) * 3.0
    score += min(row["avg_blocks"] / 12.0, 1.0) * 2.0
    score += (1.0 / max(avg_parse_ms, 0.001)) * 0.3
    score -= row["errors"] * 5.0
    return round(score, 4)


scored_benchmark_df = benchmark_df.copy()
scored_benchmark_df["rag_score"] = scored_benchmark_df.apply(score_parser_row, axis=1)
scored_benchmark_df = scored_benchmark_df.sort_values(
    ["rag_score", "errors", "docs_with_tables"], ascending=[False, True, False]
)
scored_benchmark_df


,parser,docs,errors,docs_with_headings,docs_with_tables,avg_parse_ms,total_parse_ms,avg_blocks,avg_chunks,avg_heading_count,avg_table_count,avg_plain_text_chars,rag_score
0,markdown-it-py,118,0,118,117,1.483,206.195,28.40,28.40,7.5,1.18,3632.5,8.1769
1,mistune,118,0,118,117,1.520,199.462,15.74,15.74,7.5,1.18,2851.7,8.1719
3,python-markdown,118,0,118,117,2.496,317.198,19.03,19.03,7.5,1.18,3047.1,8.0948
2,marko,118,0,118,117,5.048,617.865,15.73,15.73,7.5,1.17,2868.0,8.0340


In [21]:
best_parser = scored_benchmark_df.iloc[0]["parser"]
runner_up = scored_benchmark_df.iloc[1]["parser"]

recommendation_lines = [
    f"Recommended primary parser: {best_parser}",
    f"Runner-up: {runner_up}",
    "",
    "Decision notes:",
    "- Prefer markdown-it-py if it preserves headings and tables cleanly with acceptable runtime.",
    "- Keep mistune as the fallback when iteration speed matters more than the richest block structure.",
    "- Use marko if its AST proves easier to maintain than token-walking in later RAG notebooks.",
    "- Keep Python-Markdown as a reference parser, not the default RAG parser, unless it clearly wins on this corpus.",
]

print("\n".join(recommendation_lines))


Recommended primary parser: markdown-it-py
Runner-up: mistune

Decision notes:
- Prefer markdown-it-py if it preserves headings and tables cleanly with acceptable runtime.
- Keep mistune as the fallback when iteration speed matters more than the richest block structure.
- Use marko if its AST proves easier to maintain than token-walking in later RAG notebooks.
- Keep Python-Markdown as a reference parser, not the default RAG parser, unless it clearly wins on this corpus.
